# 02 - Silver Transformation

## Purpose

Transform raw Bronze tables into clean, typed, deduplicated, and conformed Silver Delta tables.

## Transformation Rules

- Clean column names.
- Cast dates, timestamps, booleans, and numeric fields.
- Standardize status, segment, channel, and type values.
- Remove duplicate business keys.
- Validate foreign keys before Gold modeling.
- Preserve ingestion metadata where useful for lineage.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

entities = ["customers", "accounts", "products", "branches", "transactions"]


def clean_column_names(df):
    cleaned = df
    for column_name in df.columns:
        new_name = column_name.strip().lower().replace(" ", "_").replace("-", "_")
        cleaned = cleaned.withColumnRenamed(column_name, new_name)
    return cleaned


def latest_by_key(df, key_columns):
    window_spec = Window.partitionBy(*key_columns).orderBy(F.col("_ingestion_timestamp").desc())
    return df.withColumn("_row_number", F.row_number().over(window_spec)).filter(F.col("_row_number") == 1).drop("_row_number")


def write_silver(df, entity_name):
    table_name = f"silver_{entity_name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {df.count()} rows written")

In [ ]:
# Customers
customers = clean_column_names(spark.table("bronze_customers"))
customers_silver = (
    customers
    .withColumn("customer_id", F.upper(F.trim("customer_id")))
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("last_name", F.initcap(F.trim("last_name")))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("state", F.upper(F.trim("state")))
    .withColumn("country", F.upper(F.trim("country")))
    .withColumn("customer_segment", F.initcap(F.trim("customer_segment")))
    .withColumn("customer_status", F.initcap(F.trim("customer_status")))
    .withColumn("date_of_birth", F.to_date("date_of_birth"))
    .withColumn("created_date", F.to_date("created_date"))
)
customers_silver = latest_by_key(customers_silver, ["customer_id"])
write_silver(customers_silver, "customers")

In [ ]:
# Accounts
accounts = clean_column_names(spark.table("bronze_accounts"))
accounts_silver = (
    accounts
    .withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("customer_id", F.upper(F.trim("customer_id")))
    .withColumn("product_id", F.upper(F.trim("product_id")))
    .withColumn("branch_id", F.upper(F.trim("branch_id")))
    .withColumn("account_type", F.initcap(F.trim("account_type")))
    .withColumn("account_status", F.initcap(F.trim("account_status")))
    .withColumn("opened_date", F.to_date("opened_date"))
    .withColumn("closed_date", F.to_date("closed_date"))
    .withColumn("current_balance", F.col("current_balance").cast("decimal(18,2)"))
)
accounts_silver = latest_by_key(accounts_silver, ["account_id"])
write_silver(accounts_silver, "accounts")

In [ ]:
# Products
products = clean_column_names(spark.table("bronze_products"))
products_silver = (
    products
    .withColumn("product_id", F.upper(F.trim("product_id")))
    .withColumn("product_name", F.initcap(F.trim("product_name")))
    .withColumn("product_category", F.initcap(F.trim("product_category")))
    .withColumn("product_family", F.initcap(F.trim("product_family")))
    .withColumn("fee_model", F.initcap(F.trim("fee_model")))
    .withColumn("is_active", F.lower(F.trim("is_active")).cast("boolean"))
    .withColumn("launch_date", F.to_date("launch_date"))
)
products_silver = latest_by_key(products_silver, ["product_id"])
write_silver(products_silver, "products")

In [ ]:
# Branches
branches = clean_column_names(spark.table("bronze_branches"))
branches_silver = (
    branches
    .withColumn("branch_id", F.upper(F.trim("branch_id")))
    .withColumn("branch_name", F.initcap(F.trim("branch_name")))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("state", F.upper(F.trim("state")))
    .withColumn("region", F.initcap(F.trim("region")))
    .withColumn("opened_date", F.to_date("opened_date"))
    .withColumn("is_active", F.lower(F.trim("is_active")).cast("boolean"))
)
branches_silver = latest_by_key(branches_silver, ["branch_id"])
write_silver(branches_silver, "branches")

In [ ]:
# Transactions
transactions = clean_column_names(spark.table("bronze_transactions"))
transactions_silver = (
    transactions
    .withColumn("transaction_id", F.upper(F.trim("transaction_id")))
    .withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("product_id", F.upper(F.trim("product_id")))
    .withColumn("branch_id", F.upper(F.trim("branch_id")))
    .withColumn("transaction_timestamp", F.to_timestamp("transaction_timestamp"))
    .withColumn("transaction_date", F.to_date("transaction_timestamp"))
    .withColumn("transaction_type", F.initcap(F.trim("transaction_type")))
    .withColumn("transaction_channel", F.initcap(F.trim("transaction_channel")))
    .withColumn("transaction_amount", F.col("transaction_amount").cast("decimal(18,2)"))
    .withColumn("currency", F.upper(F.trim("currency")))
    .withColumn("transaction_status", F.initcap(F.trim("transaction_status")))
)
transactions_silver = latest_by_key(transactions_silver, ["transaction_id"])
write_silver(transactions_silver, "transactions")

In [ ]:
# Business key and referential integrity validation
checks = []
checks.append(("customers_missing_key", spark.table("silver_customers").filter(F.col("customer_id").isNull()).count()))
checks.append(("accounts_missing_key", spark.table("silver_accounts").filter(F.col("account_id").isNull()).count()))
checks.append(("transactions_missing_key", spark.table("silver_transactions").filter(F.col("transaction_id").isNull()).count()))

invalid_account_customer = spark.table("silver_accounts").alias("a").join(spark.table("silver_customers").alias("c"), "customer_id", "left_anti").count()
checks.append(("accounts_without_customer", invalid_account_customer))

invalid_transaction_account = spark.table("silver_transactions").alias("t").join(spark.table("silver_accounts").alias("a"), "account_id", "left_anti").count()
checks.append(("transactions_without_account", invalid_transaction_account))

check_df = spark.createDataFrame(checks, ["check_name", "failed_row_count"])
display(check_df)

failed = check_df.filter(F.col("failed_row_count") > 0).count()
if failed > 0:
    raise ValueError("Silver validation failed. Review failed_row_count values.")